# 05 — Supervisão fraca, pseudo-rótulos e divisão segura

Este notebook cria a primeira versão dos rótulos de **oportunidade comercial** sem fingir que previsões automáticas são verdade-terreno. Ele combina dois rotuladores independentes:

1. regras comerciais apoiadas pelo vocabulário do RAG;
2. similaridade semântica com protótipos usando BERTimbau.

Somente concordâncias nos extremos semânticos são aceitas como pseudo-rótulos. Conflitos e uma amostra das concordâncias formam uma fila de auditoria humana. **Precisão, recall e F1 reais só poderão ser calculados depois da anotação humana.**

**Objetivo e conexão com o projeto.** Esta etapa cria dados de desenvolvimento provisórios ao combinar regras comerciais e similaridade semântica, mantendo separadas concordâncias automáticas, conflitos e itens destinados à revisão humana. A divisão é realizada por reunião para impedir que trechos do mesmo encontro apareçam em partições diferentes. Esse cuidado é essencial porque chunks vizinhos compartilham contexto e poderiam inflar a validação.


## 1. Decisões de execução

- O dispositivo é detectado automaticamente: RTX local, GPU do Colab ou CPU.
- O primeiro experimento usa 3.000 chunks, selecionados com semente fixa e cobertura inicial de todas as reuniões.
- O BERTimbau recebe no máximo 512 tokens, igual ao limite de sua arquitetura.
- Na RTX 3050 de 8 GB usamos batch 8 e inferência em precisão mista; na CPU usamos batch 2.
- A divisão de desenvolvimento é agrupada por `meeting_id`; nunca separamos chunks da mesma reunião.
- Ainda não criamos um conjunto de teste automático: o teste final deve conter apenas rótulos humanos.
- Nenhum trecho de transcrição é impresso nas saídas do notebook.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations

import json
import random
import re
import time
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
INITIAL_SAMPLE_SIZE = 3_000
AUDIT_SIZE = 150
MAX_LENGTH = 512

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2


### Função auxiliar: `find_project_root`

Esta célula isola a responsabilidade implementada por `find_project_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada. No Colab, entre na pasta Wedjat antes de executar.")


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `CHUNKS_PATH`, `KB_PATH`, `PSEUDO_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [1]:
ROOT = find_project_root()
CHUNKS_PATH = ROOT / "data/processed/chunks_bertimbau.jsonl"
KB_PATH = ROOT / "data/knowledge_base/totvs_rag_kb_v1.json"
PSEUDO_PATH = ROOT / "data/processed/pseudo_labels_opportunity.jsonl"
AUDIT_PATH = ROOT / "data/processed/annotation_queue_opportunity.jsonl"
SUMMARY_PATH = ROOT / "reports/metrics/weak_supervision_summary.json"

device_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print({"device": str(DEVICE), "device_name": device_name, "batch_size": BATCH_SIZE})


c:\Users\Gabriel Pereira\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cuda', 'device_name': 'NVIDIA GeForce RTX 3050', 'batch_size': 8}


## 3. Amostra reproduzível

Primeiro escolhemos um chunk aleatório por reunião. As vagas restantes são preenchidas aleatoriamente entre os demais chunks. Isso evita que as reuniões mais longas dominem toda a primeira rodada, sem impedir que algumas contribuam com mais de um exemplo.

A seleção reproduzível controla quais exemplos entram no experimento e mantém cobertura por reunião. A semente fixa permite repetir a mesma análise e separar mudanças metodológicas de variações aleatórias da amostra.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


### Preparação dos objetos desta etapa

A célula prepara `all_chunks`, `by_meeting`, `rng`, `sample` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [2]:
all_chunks = read_jsonl(CHUNKS_PATH)
by_meeting: dict[str, list[dict]] = defaultdict(list)
for row in all_chunks:
    by_meeting[str(row["meeting_id"])].append(row)

rng = random.Random(SEED)
sample = [rng.choice(rows) for _, rows in sorted(by_meeting.items())]
selected_ids = {row["chunk_id"] for row in sample}
remaining = [row for row in all_chunks if row["chunk_id"] not in selected_ids]
target_size = min(INITIAL_SAMPLE_SIZE, len(all_chunks))
if len(sample) < target_size:
    sample.extend(rng.sample(remaining, target_size - len(sample)))
sample.sort(key=lambda row: row["chunk_id"])

print({
    "total_chunks": len(all_chunks),
    "sample_chunks": len(sample),
    "meetings_in_sample": len({row["meeting_id"] for row in sample}),
})


{'total_chunks': 29972, 'sample_chunks': 3000, 'meetings_in_sample': 1126}


## 4. Rotulador 1 — regras comerciais apoiadas pelo RAG

Os chunks do RAG dos tipos `sinal_oportunidade` e `dor_para_produto` fornecem vocabulário de produtos e dores. Esse vocabulário só conta como contexto: para marcar oportunidade positiva, também exigimos intenção, compra, implantação ou uma dor explícita. Regras incertas se abstêm (`-1`).

Este rotulador produz apenas um sinal provisório. Os casos de abstenção e conflito são preservados porque forçar uma classe nesses exemplos aumentaria o ruído e daria uma falsa impressão de verdade-terreno.


In [ ]:
def normalize_for_match(text: str) -> str:
    text = unicodedata.normalize("NFKD", text.lower())
    return "".join(char for char in text if not unicodedata.combining(char))


### Preparação dos objetos desta etapa

A célula prepara `knowledge_base`, `rag_terms`, `PRODUCT_PATTERN`, `INTENT_PATTERN` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
knowledge_base = json.loads(KB_PATH.read_text(encoding="utf-8"))
rag_terms: set[str] = set()
for document in knowledge_base:
    if document.get("document_type") in {"sinal_oportunidade", "dor_para_produto", "produto"}:
        candidates = [document.get("product", ""), *document.get("keywords", [])]
        rag_terms.update(normalize_for_match(term) for term in candidates if len(str(term).strip()) >= 4)

PRODUCT_PATTERN = re.compile(
    r"\b(?:erp|crm|software|sistema|solucao|plataforma|protheus|fluig|datasul|rm|totvs|folha|hcm|wms|licenca|modulo)\b"
)
INTENT_PATTERN = re.compile(
    r"\b(?:precisamos?|necessitamos?|queremos?|gostariamos?|buscamos?|procuramos?|avaliar|avaliando|interesse|interessados?)\b"
)
BUY_PATTERN = re.compile(
    r"\b(?:proposta|cotacao|orcamento|preco|valor|contratar|adquirir|comprar|implantacao|implementar|migrar|substituir|prazo)\b"
)
PAIN_PATTERN = re.compile(
    r"\b(?:problema|dificuldade|gargalo|retrabalho|manual|planilha|integracao|lentidao|erro|nao atende|limitacao)\b"
)
REFUSAL_PATTERN = re.compile(
    r"\b(?:sem interesse|nao temos interesse|nao precisamos|nao pretendemos|nao vamos contratar|projeto cancelado|projeto descartado)\b"
)


### Funções auxiliares da seção

Esta célula agrupa `rag_context_hits`, `rule_label`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def rag_context_hits(normalized_text: str) -> list[str]:
    return sorted(term for term in rag_terms if len(term) >= 5 and term in normalized_text)[:20]

def rule_label(text: str) -> tuple[int, int, list[str]]:
    normalized = normalize_for_match(text)
    if REFUSAL_PATTERN.search(normalized):
        return 0, 4, ["recusa_explicita"]

    signals = {
        "intencao": bool(INTENT_PATTERN.search(normalized)),
        "compra_implantacao": bool(BUY_PATTERN.search(normalized)),
        "dor": bool(PAIN_PATTERN.search(normalized)),
        "produto": bool(PRODUCT_PATTERN.search(normalized)),
        "contexto_rag": bool(rag_context_hits(normalized)),
    }
    score = 2 * signals["intencao"] + 2 * signals["compra_implantacao"] + signals["dor"] + signals["produto"] + signals["contexto_rag"]
    reasons = [name for name, present in signals.items() if present]

    if score >= 4 and (signals["intencao"] or signals["compra_implantacao"]):
        return 1, score, reasons
    if not (signals["intencao"] or signals["compra_implantacao"] or signals["dor"]):
        return 0, score, ["sem_intencao_compra_ou_dor"]
    return -1, score, reasons


### Preparação dos objetos desta etapa

A célula prepara `rule_results` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [3]:
rule_results = [rule_label(row["text"]) for row in sample]
print({"rule_labels": dict(Counter(label for label, _, _ in rule_results))})


{'rule_labels': {1: 814, 0: 1272, -1: 914}}


## 5. Rotulador 2 — BERTimbau por protótipos

O BERTimbau base ainda não é um classificador de oportunidades. Aqui ele transforma textos e exemplos prototípicos em vetores; a decisão usa a diferença entre a similaridade com protótipos positivos e negativos. É um sinal semântico inicial, não o fine-tuning final da Sprint 3. Para reduzir ruído, apenas os 30% mais negativos e os 30% mais positivos recebem rótulo; a faixa central se abstém.

Este rotulador produz apenas um sinal provisório. Os casos de abstenção e conflito são preservados porque forçar uma classe nesses exemplos aumentaria o ruído e daria uma falsa impressão de verdade-terreno.


In [ ]:
POSITIVE_PROTOTYPES = [
    "O cliente precisa de um novo sistema e quer receber uma proposta comercial.",
    "A empresa está avaliando migrar o ERP e pediu preço e prazo de implantação.",
    "Há interesse em contratar uma solução para eliminar processos manuais.",
    "O cliente relatou uma dor de integração e quer conhecer um produto TOTVS.",
    "A organização pretende substituir o fornecedor atual e solicitou uma cotação.",
    "Existe orçamento aprovado para implantar o módulo no próximo trimestre.",
]
NEGATIVE_PROTOTYPES = [
    "Os participantes apenas se cumprimentam e aguardam o início da reunião.",
    "A conversa trata somente de agenda, horário e presença dos convidados.",
    "O participante explica um procedimento sem pedir produto, proposta ou contratação.",
    "Não existe interesse em comprar ou trocar o sistema neste momento.",
    "A equipe encerra a reunião e combina o envio da ata.",
    "O trecho é neutro e não apresenta necessidade comercial do cliente.",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()


### Geração de representações semânticas

A função processa textos em lotes, aplica mean pooling com máscara e normalização L2. O resultado permite comparar chunks e protótipos por similaridade.


In [ ]:
def embed_texts(texts: list[str], batch_size: int = BATCH_SIZE) -> torch.Tensor:
    vectors: list[torch.Tensor] = []
    started = time.perf_counter()
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)
        with torch.inference_mode():
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                hidden = model(**encoded).last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1)
            pooled = (hidden.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            pooled = F.normalize(pooled, p=2, dim=1)
        vectors.append(pooled.cpu())
    elapsed = time.perf_counter() - started
    return torch.cat(vectors), elapsed


### Preparação dos objetos desta etapa

A célula prepara `positive_centroid`, `negative_centroid`, `margins`, `bert_labels` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [4]:
prototype_vectors, prototype_seconds = embed_texts(POSITIVE_PROTOTYPES + NEGATIVE_PROTOTYPES)
positive_centroid = F.normalize(prototype_vectors[:len(POSITIVE_PROTOTYPES)].mean(dim=0), dim=0)
negative_centroid = F.normalize(prototype_vectors[len(POSITIVE_PROTOTYPES):].mean(dim=0), dim=0)
chunk_vectors, embedding_seconds = embed_texts([row["text"] for row in sample])
margins = (chunk_vectors @ positive_centroid - chunk_vectors @ negative_centroid).numpy()
low_threshold, high_threshold = np.quantile(margins, [0.30, 0.70]).tolist()
bert_labels = np.where(margins <= low_threshold, 0, np.where(margins >= high_threshold, 1, -1))

del model, chunk_vectors
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print({
    "bert_labels": dict(Counter(int(label) for label in bert_labels)),
    "margin_p30": round(low_threshold, 6),
    "margin_p70": round(high_threshold, 6),
    "embedding_seconds": round(embedding_seconds, 2),
})


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11645.81it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'bert_labels': {1: 900, -1: 1200, 0: 900}, 'margin_p30': -0.00088, 'margin_p70': 0.019753, 'embedding_seconds': 33.13}


## 6. Consolidação, fila humana e split de desenvolvimento

Uma concordância significa que os dois métodos deram o mesmo rótulo sem abstenção. Esse conjunto pode iniciar os experimentos, mas continua sendo `pseudo_label`. A fila humana é cega: contém texto e campos vazios de anotação, sem mostrar as previsões, para reduzir viés do revisor. Cada item vem de uma reunião diferente e todas essas reuniões ficam fora do desenvolvimento. O split `train/validation` é apenas de desenvolvimento; o teste final nascerá da auditoria humana.

Nesta seção, os sinais anteriores são reunidos e convertidos nos artefatos consumidos pela modelagem. A reserva para auditoria humana permanece isolada do desenvolvimento para evitar contaminação da avaliação futura.


In [ ]:
combined: list[dict] = []
for row, (rule_value, rule_score, rule_reasons), bert_value, margin in zip(sample, rule_results, bert_labels, margins):
    bert_value = int(bert_value)
    agreement = rule_value in {0, 1} and rule_value == bert_value
    combined.append({
        "meeting_id": str(row["meeting_id"]),
        "chunk_id": row["chunk_id"],
        "chunk_index": row["chunk_index"],
        "text": row["text"],
        "rule_label": rule_value,
        "rule_score": rule_score,
        "rule_reasons": rule_reasons,
        "bert_prototype_label": bert_value,
        "bert_margin": round(float(margin), 8),
        "pseudo_label": rule_value if agreement else None,
        "label_origin": "rule_and_bert_agreement" if agreement else "needs_review",
    })

pseudo_rows_all = [row for row in combined if row["pseudo_label"] is not None]
needs_review = [row for row in combined if row["pseudo_label"] is None]


### Reserva de exemplos por reunião

A seleção aceita no máximo um item por reunião para a fila humana. Isso amplia a diversidade da auditoria e mantém essas reuniões fora do desenvolvimento.


In [ ]:
def take_unique_meetings(
    rows: list[dict], predicate, amount: int, rng: random.Random, used_meetings: set[str]
) -> list[dict]:
    candidates = [row for row in rows if predicate(row)]
    rng.shuffle(candidates)
    selected = []
    for row in candidates:
        if row["meeting_id"] in used_meetings:
            continue
        selected.append(row)
        used_meetings.add(row["meeting_id"])
        if len(selected) == amount:
            break
    return selected


### Verificações de integridade

As asserções tornam explícitas as condições que precisam permanecer verdadeiras para que os resultados sejam considerados consistentes.


In [ ]:
audit_rng = random.Random(SEED + 1)
per_group = AUDIT_SIZE // 3
reserved_meetings: set[str] = set()
audit_candidates = []
audit_candidates += take_unique_meetings(
    pseudo_rows_all, lambda row: row["pseudo_label"] == 1, per_group, audit_rng, reserved_meetings
)
audit_candidates += take_unique_meetings(
    pseudo_rows_all, lambda row: row["pseudo_label"] == 0, per_group, audit_rng, reserved_meetings
)
audit_candidates += take_unique_meetings(
    needs_review, lambda row: True, per_group, audit_rng, reserved_meetings
)
if len(audit_candidates) < AUDIT_SIZE:
    audit_candidates += take_unique_meetings(
        combined, lambda row: True, AUDIT_SIZE - len(audit_candidates), audit_rng, reserved_meetings
    )
audit_rng.shuffle(audit_candidates)
audit_rows = [{
    "meeting_id": row["meeting_id"],
    "chunk_id": row["chunk_id"],
    "text": row["text"],
    "human_label": None,
    "reviewer_notes": "",
} for row in audit_candidates[:AUDIT_SIZE]]
audit_meetings = {row["meeting_id"] for row in audit_rows}
assert len(audit_meetings) == len(audit_rows)

pseudo_rows = [row for row in pseudo_rows_all if row["meeting_id"] not in audit_meetings]
meeting_targets: dict[str, int] = defaultdict(int)


### Divisão agrupada de desenvolvimento

As reuniões são separadas antes de atribuir cada chunk ao treino ou à validação. As asserções seguintes confirmam que não existe sobreposição entre as partições.


In [ ]:
for row in pseudo_rows:
    meeting_targets[row["meeting_id"]] = max(meeting_targets[row["meeting_id"]], row["pseudo_label"])
meeting_ids = sorted(meeting_targets)
strata = [meeting_targets[meeting_id] for meeting_id in meeting_ids]
stratify = strata if len(set(strata)) == 2 and min(Counter(strata).values()) >= 2 else None
train_meetings, validation_meetings = train_test_split(
    meeting_ids, test_size=0.20, random_state=SEED, stratify=stratify
)
train_meetings, validation_meetings = set(train_meetings), set(validation_meetings)
assert train_meetings.isdisjoint(validation_meetings)
assert (train_meetings | validation_meetings).isdisjoint(audit_meetings)
for row in pseudo_rows:
    row["weak_split"] = "train" if row["meeting_id"] in train_meetings else "validation"

PSEUDO_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
with PSEUDO_PATH.open("w", encoding="utf-8") as file:
    for row in pseudo_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")
with AUDIT_PATH.open("w", encoding="utf-8") as file:
    for row in audit_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

co_labeled = [row for row in combined if row["rule_label"] in {0, 1} and row["bert_prototype_label"] in {0, 1}]


### Consolidação do relatório da etapa

Configurações, contagens, métricas e alertas são reunidos em um resumo auditável. O relatório registra resultados agregados sem acrescentar novas transformações aos dados.


In [ ]:
summary = {
    "seed": SEED,
    "model_name": MODEL_NAME,
    "device": str(DEVICE),
    "device_name": device_name,
    "torch_version": torch.__version__,
    "cuda_build": torch.version.cuda,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "sample_chunks": len(sample),
    "sample_meetings": len({row["meeting_id"] for row in sample}),
    "rule_label_counts": dict(Counter(str(row["rule_label"]) for row in combined)),
    "bert_label_counts": dict(Counter(str(row["bert_prototype_label"]) for row in combined)),
    "bert_margin_p30": low_threshold,
    "bert_margin_p70": high_threshold,
    "pseudo_label_counts_before_audit_reserve": dict(Counter(str(row["pseudo_label"]) for row in pseudo_rows_all)),
    "pseudo_label_count_before_audit_reserve": len(pseudo_rows_all),
    "pseudo_label_coverage": len(pseudo_rows_all) / len(combined),
    "development_pseudo_label_counts": dict(Counter(str(row["pseudo_label"]) for row in pseudo_rows)),
    "development_pseudo_label_count": len(pseudo_rows),
    "agreement_among_co_labeled": (len(pseudo_rows_all) / len(co_labeled)) if co_labeled else None,
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "split_leakage_meetings": len(train_meetings & validation_meetings),
    "audit_queue_size": len(audit_rows),
    "audit_meetings": len(audit_meetings),
    "audit_development_leakage_meetings": len(audit_meetings & (train_meetings | validation_meetings)),
    "prototype_embedding_seconds": prototype_seconds,
    "chunk_embedding_seconds": embedding_seconds,
    "metric_warning": "Agreement and coverage are not precision. Final metrics require human ground truth.",
}


### Persistência dos artefatos

Os resultados desta etapa são gravados nos caminhos definidos para consumo posterior. O conteúdo persistido segue o contrato já estabelecido pelo projeto.


In [6]:
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print({
    "pseudo_labels_before_audit_reserve": len(pseudo_rows_all),
    "development_pseudo_labels": len(pseudo_rows),
    "development_class_balance": summary["development_pseudo_label_counts"],
    "coverage": round(summary["pseudo_label_coverage"], 4),
    "agreement_among_co_labeled": round(summary["agreement_among_co_labeled"], 4) if summary["agreement_among_co_labeled"] is not None else None,
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "audit_queue": len(audit_rows),
    "split_leakage": summary["split_leakage_meetings"],
    "audit_development_leakage": summary["audit_development_leakage_meetings"],
})


{'pseudo_labels_before_audit_reserve': 906, 'development_pseudo_labels': 679, 'development_class_balance': {'1': 276, '0': 403}, 'coverage': 0.302, 'agreement_among_co_labeled': 0.7007, 'train_meetings': 388, 'validation_meetings': 98, 'audit_queue': 150, 'split_leakage': 0, 'audit_development_leakage': 0}


## 7. Como interpretar e continuar

- `pseudo_labels_opportunity.jsonl` serve para prototipar TF-IDF + Logistic Regression e o fine-tuning do BERTimbau. O campo `label_origin` impede confusão com rótulos humanos.
- `annotation_queue_opportunity.jsonl` deve ser preenchido por uma pessoa com `human_label = 0` ou `1`. Casos ambíguos podem receber `null` e uma observação.
- A primeira avaliação honesta é comparar os pseudo-rótulos com essa auditoria humana e calcular precisão por origem/classe.
- Depois da auditoria, congelaremos reuniões humanas exclusivas para teste e treinaremos os dois modelos exigidos pela Sprint 3 no mesmo split.
- Similaridade de BERTimbau base não substitui fine-tuning; ela apenas reduz o custo de iniciar a anotação.

A leitura dos resultados deve considerar tanto o padrão observado quanto as limitações da referência. As conclusões desta seção orientam a próxima etapa, mas não substituem a validação humana prevista no projeto.
